# Code Ownership, notebook 3: the encoders and the feature sets

Frozen encoders (no weights are changed), mean-pooled and L2-normalised: UniXcoder-base and CodeBERT-base (the proposal's choices) and all-mpnet-base-v2 (a general text
encoder as a control). Feature sets A-D (answer; + similarities; + answer-times-reference; joint question+answer) with and without the hand-built features.

In [ ]:
import sys, warnings, json, pickle
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../../helpers").resolve()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
import ko_data as K, ko_models as M
from core import metrics, group_bootstrap, paired_diff_bootstrap, qwk, band
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = K.load()
dev = d[d.role == "dev"].reset_index(drop=True)
test = d[d.role == "test"].reset_index(drop=True)
y, groups, folds = dev.score.to_numpy(), dev.snippet_id.to_numpy(), dev.fold.to_numpy()
def ci(p, stat):
    return group_bootstrap(y, p, groups, stat, n=1000)
def sp(a, b):
    return spearmanr(a, b).correlation
def mae(a, b):
    return float(np.mean(np.abs(a - b)))
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [ ]:
import time
from embedder import Encoder
available = {}
for key in ("unixcoder", "codebert", "mpnet"):
    local = Encoder._local(key)
    available[key] = local or "hub cache"
print(available)
ENCODERS = [k for k in ("unixcoder", "codebert", "mpnet") if (Encoder._local(k) or k == "mpnet")]
print("encoders used:", ENCODERS)

In [ ]:
H = K.hand_features(dev)
info = []
EMB = {}
for key in ENCODERS:
    t = time.time(); E = K.embeddings(key, dev); EMB[key] = E
    sets = K.feature_sets(E, H)
    info.append({"encoder": key, "embedding dim": E["ans"].shape[1], **{f"dim {s}": v.shape[1] for s, v in sets.items()}, "seconds": round(time.time() - t, 1)})
pd.DataFrame(info).set_index("encoder")

A first look: cosine similarity between the answer and the reference explanation, by written level. If the embeddings carried the 'is it right' signal this would climb with the level.

In [ ]:
rows = []
for key, E in EMB.items():
    for part in ("ref", "code", "ques"):
        c = (E["ans"] * E[part]).sum(1)
        rows.append({"encoder": key, "answer vs": part, "Spearman with level": round(sp(c, dev.rating), 3), **{f"level {r}": round(float(c[dev.rating == r].mean()), 3) for r in range(1, 6)}})
pd.DataFrame(rows)

## Reading the results

_Reading of the results: written after the run._